# RetinoCheck AI — Regression vs Classification Benchmark

**Research only. Never a medical device or medical advice.**

This notebook compares **ResNet-18** and **EfficientNet-B0** across two training paradigms:
1. **5-Class Classification**: Standard cross-entropy loss over 5 logits with class-weighted loss for imbalance.
2. **Ordinal Regression**: Single scalar output trained with SmoothL1 (or MSE) loss, paired with **4 validation-optimized decision thresholds** to maximize Quadratic Weighted Kappa (QWK).

All metrics are compiled and saved to `reports/results.csv` with columns:
`model, mode, regularization, accuracy, macro_f1, qwk`.


## 1 · Environment Setup
Mounts Drive, clones the repository, installs pinned dependencies, and checks GPU availability.

In [ ]:
# ── Colab Setup ────────────────────────────────────────────────────────────
from google.colab import drive
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # Replace with your repository URL
REPO_DIR = Path("/content/retinocheck-ai")
DRIVE_ROOT = Path("/content/drive/MyDrive/RetinoCheckAI")

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository URL first.")

drive.mount("/content/drive")
for folder in ("checkpoints", "reports"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
print("Setup complete. GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2 · Why Ordinal Regression with 4 Optimized Thresholds?

- **Ordinal Nature of Diabetic Retinopathy**: Severity grades follow a clinical continuum ($0 < 1 < 2 < 3 < 4$). Standard cross-entropy penalizes a misclassification of grade 0 as grade 1 identically to grade 0 as grade 4. Regression loss penalizes larger grade errors proportionally.
- **The 4 Decision Thresholds**: For 5 grades, 4 cut-points $[\theta_0, \theta_1, \theta_2, \theta_3]$ partition continuous predictions into discrete integers:
  - Grade 0: $\hat{y} < \theta_0$
  - Grade 1: $\theta_0 \le \hat{y} < \theta_1$
  - Grade 2: $\theta_1 \le \hat{y} < \theta_2$
  - Grade 3: $\theta_2 \le \hat{y} < \theta_3$
  - Grade 4: $\hat{y} \ge \theta_3$
- Standard rounding corresponds to fixed cut-points $[0.5, 1.5, 2.5, 3.5]$.
- **Nelder-Mead Optimization**: We optimize $[\theta_0, \theta_1, \theta_2, \theta_3]$ directly on validation continuous outputs to maximize QWK, adapting to severe class imbalance.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch

from src.config import Config, CHECKPOINT_ROOT
from src.train import train
from src.evaluate import evaluate, save_results_csv

print("PyTorch version:", torch.__version__)


## 3 · Run the 4 Benchmark Experiments

We train both backbones under identical data splits, learning rate schedule (cosine), and AdamW L2 weight decay ($0.01$):
1. **ResNet-18** (Classification)
2. **ResNet-18** (Regression + Threshold Search)
3. **EfficientNet-B0** (Classification)
4. **EfficientNet-B0** (Regression + Threshold Search)


In [ ]:
EXPERIMENTS = [
    {
        "model": "resnet18",
        "mode": "classification",
        "regularization": "L2 (wd=0.01)",
        "output_dir": str(CHECKPOINT_ROOT / "resnet18_classification"),
        "regression_loss": "smooth_l1",
    },
    {
        "model": "resnet18",
        "mode": "regression",
        "regularization": "L2 (wd=0.01)",
        "output_dir": str(CHECKPOINT_ROOT / "resnet18_regression"),
        "regression_loss": "smooth_l1",
    },
    {
        "model": "efficientnet_b0",
        "mode": "classification",
        "regularization": "L2 (wd=0.01)",
        "output_dir": str(CHECKPOINT_ROOT / "efficientnet_b0_classification"),
        "regression_loss": "smooth_l1",
    },
    {
        "model": "efficientnet_b0",
        "mode": "regression",
        "regularization": "L2 (wd=0.01)",
        "output_dir": str(CHECKPOINT_ROOT / "efficientnet_b0_regression"),
        "regression_loss": "smooth_l1",
    },
]

results = []

for exp in EXPERIMENTS:
    print(f"\n{'=' * 65}")
    print(f"Starting: {exp['model']} [{exp['mode']}]")
    print(f"Output  : {exp['output_dir']}")
    print(f"{'=' * 65}")
    
    cfg = Config(
        model_name=exp["model"],
        mode=exp["mode"],
        regression_loss=exp["regression_loss"],
        output_dir=exp["output_dir"],
        epochs=15,
        batch_size=32,
        lr=1e-4,
        weight_decay=1e-2,
        threshold_search=True,
        patience=5,
        seed=42,
    )
    
    best_ckpt = Path(exp["output_dir"]) / "best.pt"
    if not best_ckpt.exists():
        train(cfg)
    else:
        print(f"Reusing existing checkpoint: {best_ckpt}")
        
    # Evaluate on held-out test split
    test_eval = evaluate(str(best_ckpt), output_dir=f"{exp['output_dir']}/reports", split="test")
    
    results.append({
        "model": exp["model"],
        "mode": exp["mode"],
        "regularization": exp["regularization"],
        "accuracy": test_eval["accuracy"],
        "macro_f1": test_eval["macro_f1"],
        "qwk": test_eval["qwk"],
    })
    print(f"Test QWK: {test_eval['qwk']:.4f} | Acc: {test_eval['accuracy']:.4f} | F1: {test_eval['macro_f1']:.4f}")


## 4 · Save to `reports/results.csv` and Compare Table

Writes the standard comparative table to `reports/results.csv` and displays it.

In [ ]:
csv_path = Path("reports/results.csv")
save_results_csv(results, str(csv_path))
print(f"Successfully saved comparison to: {csv_path.resolve()}")

df = pd.DataFrame(results)
display(df)


## 5 · Comparative Visualization

Bar plots comparing Quadratic Weighted Kappa (primary metric), Macro F1, and Accuracy across models and training modes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [("qwk", "Quadratic Weighted Kappa (QWK)"), ("macro_f1", "Macro F1 Score"), ("accuracy", "Accuracy")]

labels = [f"{r['model']}\n({r['mode']})" for r in results]
colors = ["#1f77b4" if r["mode"] == "classification" else "#2ca02c" for r in results]

for ax, (metric_key, title) in zip(axes, metrics):
    vals = [r[metric_key] for r in results]
    bars = ax.bar(labels, vals, color=colors, width=0.55)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(metric_key.upper())
    ax.grid(axis="y", alpha=0.3)
    ax.bar_label(bars, fmt="%.3f", padding=3)
    ax.set_ylim(0, max(vals) * 1.2 if max(vals) > 0 else 1.0)

fig.suptitle("Architecture & Objective Comparison: Classification vs Ordinal Regression", fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig("reports/model_mode_comparison.png", dpi=160, bbox_inches="tight")
plt.show()


## 6 · Inspecting Optimized Decision Thresholds

Displays the 4 learned decision cut-points from both regression models compared to default rounding ($[0.5, 1.5, 2.5, 3.5]$).

In [ ]:
for exp in EXPERIMENTS:
    if exp["mode"] == "regression":
        ckpt_path = Path(exp["output_dir"]) / "best.pt"
        if ckpt_path.exists():
            ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=True)
            thr = ckpt.get("best_thresholds")
            print(f"\n{exp['model']} Optimized 4 Thresholds:")
            if thr is not None:
                print("  Learned Cut-points:", [round(t, 4) for t in thr])
                print("  Default Rounding  :", [0.5, 1.5, 2.5, 3.5])
            else:
                print("  Standard rounding [0.5, 1.5, 2.5, 3.5] retained (achieved highest validation QWK).")
